# Stage 4 -- Onboarding Intelligence Room Walkthrough

**Book**: *Mastering Agentic AI for Customer Journey Marketing*  
**Author**: Pushparajan Ramar  
**Chapter**: 9  
**Framework**: AutoGen (pyautogen >= 0.4)

---

This notebook walks through the **Onboarding Intelligence Room** step by step:

1. Explore the product-usage tools
2. Explore the milestone and support-ticket tools
3. Instantiate each agent individually
4. Run the full RoundRobinGroupChat
5. Inspect the final onboarding action plan

## 0. Setup

In [ ]:
import os, sys, json

# Ensure the project root is on sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Force mock mode so the notebook runs without real API keys
os.environ["USE_MOCK"] = "true"
print(f"Project root: {project_root}")
print(f"USE_MOCK: {os.environ['USE_MOCK']}")

## 1. Product-Usage Tools

These tools provide usage analytics that the **Product Specialist** agent uses
to identify the *aha* feature.

In [ ]:
from stage4_onboarding.tools.product_usage_tools import (
    get_usage_heatmap,
    get_adoption_score,
    get_feature_completion_rate,
    identify_friction_points,
)

# Smooth customer
print("=== CUST-1001 (Smooth) ===")
print("\nUsage Heatmap:")
print(get_usage_heatmap("CUST-1001", 7))

In [ ]:
print("Feature Completion Rate:")
print(get_feature_completion_rate("CUST-1001"))

print("\nFriction Points:")
print(identify_friction_points("CUST-1001"))

In [ ]:
# Struggling customer
print("=== CUST-1002 (Struggling) ===")
print("\nUsage Heatmap:")
print(get_usage_heatmap("CUST-1002", 7))

print("\nFriction Points:")
print(identify_friction_points("CUST-1002"))

## 2. Milestone and Support-Ticket Tools

These tools feed the **Customer Success Agent**.

In [ ]:
from stage4_onboarding.tools.onboarding_milestone_tools import (
    get_onboarding_checklist,
    get_time_to_value,
    update_milestone_status,
)

print("Standard Onboarding Checklist:")
print(get_onboarding_checklist("standard"))

In [ ]:
print("\nTime-to-Value — CUST-1001:")
print(get_time_to_value("CUST-1001"))

print("\nTime-to-Value — CUST-1002:")
print(get_time_to_value("CUST-1002"))

In [ ]:
from stage4_onboarding.tools.support_ticket_tools import (
    get_recent_tickets,
    get_sentiment_score,
    create_onboarding_task,
)

print("Recent Tickets — CUST-1002:")
print(get_recent_tickets("CUST-1002"))

print("\nSentiment — CUST-1002:")
print(get_sentiment_score("CUST-1002"))

In [ ]:
# Create a sample task
print("Create Onboarding Task:")
result = create_onboarding_task("CUST-1002", "Urgent: personal outreach for struggling customer", "Sarah Kim")
print(result)

## 3. Instantiate Agents

Each agent is created via a factory function. In mock mode, the model client
is configured but will only be called if you have a valid API key.

In [ ]:
from stage4_onboarding.agents.product_specialist import create_product_specialist
from stage4_onboarding.agents.cs_agent import create_cs_agent
from stage4_onboarding.agents.onboarding_coordinator import create_onboarding_coordinator

ps = create_product_specialist()
cs = create_cs_agent()
coord = create_onboarding_coordinator()

print(f"Product Specialist : {ps.name}")
print(f"  Tools: {[t.__name__ for t in [ps._tools[i] for i in range(len(ps._tools))]] if hasattr(ps, '_tools') else 'see agent config'}")
print(f"CS Agent           : {cs.name}")
print(f"Coordinator        : {coord.name}")

## 4. Run the Full Onboarding Intelligence Room (Mock Mode)

We use the deterministic mock runner so the notebook works without an LLM API key.

In [ ]:
from stage4_onboarding.project_onboarding_room.main import _mock_onboarding_plan

smooth_profile = {
    "customer_id": "CUST-1001",
    "company": "TechNova Solutions",
    "industry": "SaaS / Technology",
    "plan": "standard",
    "signup_date": "2026-03-28",
    "days_since_signup": 10,
    "contact_name": "Alice Chen",
    "contact_role": "VP of Marketing",
    "team_size": 8,
    "goals": "Centralise marketing analytics and automate weekly reports",
    "scenario": "SMOOTH ONBOARDING",
}

plan_smooth = _mock_onboarding_plan(smooth_profile)

In [ ]:
struggling_profile = {
    "customer_id": "CUST-1002",
    "company": "RetailEdge Inc.",
    "industry": "Retail / E-commerce",
    "plan": "standard",
    "signup_date": "2026-03-25",
    "days_since_signup": 13,
    "contact_name": "Bob Martinez",
    "contact_role": "Marketing Manager",
    "team_size": 3,
    "goals": "Track campaign ROI across channels",
    "scenario": "STRUGGLING ONBOARDING",
}

plan_struggling = _mock_onboarding_plan(struggling_profile)

## 5. Compare the Two Plans

In [ ]:
import pandas as pd

comparison = pd.DataFrame([
    {
        "Customer": plan_smooth["customer_id"],
        "Aha Feature": plan_smooth["aha_feature"],
        "Risk Flags": len(plan_smooth["risk_flags"]),
        "Activation Steps": len(plan_smooth["activation_sequence"]),
        "Check-in Date": plan_smooth["first_check_in_date"],
        "Owner": plan_smooth["owner"],
    },
    {
        "Customer": plan_struggling["customer_id"],
        "Aha Feature": plan_struggling["aha_feature"],
        "Risk Flags": len(plan_struggling["risk_flags"]),
        "Activation Steps": len(plan_struggling["activation_sequence"]),
        "Check-in Date": plan_struggling["first_check_in_date"],
        "Owner": plan_struggling["owner"],
    },
])

comparison

In [ ]:
print("\n=== Risk Flags for CUST-1002 (Struggling) ===")
for i, flag in enumerate(plan_struggling["risk_flags"], 1):
    print(f"  {i}. {flag}")

print("\n=== Recommended Content for CUST-1002 ===")
for item in plan_struggling["recommended_content"]:
    print(f"  - {item}")

## 6. Visualise Adoption Scores

In [ ]:
import matplotlib.pyplot as plt

customers = ["CUST-1001\n(Smooth)", "CUST-1002\n(Struggling)"]
adoption_scores = [0.72, 0.18]
sentiment_scores = [0.82, 0.31]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

colors = ["#2ecc71" if s >= 0.5 else "#e74c3c" for s in adoption_scores]
axes[0].bar(customers, adoption_scores, color=colors)
axes[0].set_ylim(0, 1)
axes[0].set_title("Adoption Score")
axes[0].axhline(y=0.3, color="orange", linestyle="--", label="Risk threshold")
axes[0].legend()

colors2 = ["#2ecc71" if s >= 0.5 else "#e74c3c" for s in sentiment_scores]
axes[1].bar(customers, sentiment_scores, color=colors2)
axes[1].set_ylim(0, 1)
axes[1].set_title("Sentiment Score")
axes[1].axhline(y=0.5, color="orange", linestyle="--", label="Neutral threshold")
axes[1].legend()

plt.tight_layout()
plt.suptitle("Onboarding Health Comparison", y=1.02, fontsize=14)
plt.show()

## 7. Running with a Real LLM (Optional)

If you have an OpenAI API key, uncomment the cell below to run the full
LLM-backed group chat.

In [ ]:
# Uncomment to run with a real LLM:
# os.environ["OPENAI_API_KEY"] = "sk-..."
# from stage4_onboarding.groupchats.onboarding_intelligence_room import run_onboarding_analysis
# plan = await run_onboarding_analysis(smooth_profile, dispatch=True, verbose=True)
# print(json.dumps(plan, indent=2))

---

**End of walkthrough.** Refer to `project_onboarding_room/main.py` for the
full entry-point script that processes both sample customers.